# Анализ ожидаемой продолжительности жизни

Два набора данных, исследующих глобальную продолжительность жизни:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **Ожидаемая продолжительность жизни ВОЗ (WHO)** (2000–2015): 193 страны, 22 показателя (смертность, ИМТ/BMI, ВВП, образование и др.)

Эта книга демонстрирует импорт и анализ файлов CSV на **Python** и **R**.

## 1. Настройка: установка пакетов и загрузка наборов данных

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('Установлены pandas + plotly')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Уже существует: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Загружено {name}: {lines} строк")

## 2. Gapminder: Исследование на Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Размерность: {gap.shape}")
print(f"Континенты: {sorted(gap['continent'].unique())}")
print(f"Диапазон лет: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Ожидаемая продолжительность жизни во времени по континентам
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Ожидаемая продолжительность жизни по континентам (1952-2007)',
              labels={'lifeExp': 'Ожидаемая продолжительность жизни (лет)', 'year': 'Год'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# ВВП против ожидаемой продолжительности жизни (2007), размер пузырька = население
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='ВВП против ожидаемой продолжительности жизни (2007)',
                 labels={'gdpPercap': 'ВВП на душу населения (лог.)', 'lifeExp': 'Ожидаемая продолжительность жизни'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: Исследование на R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Распределение продолжительности жизни по континентам (boxplot)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Ожидаемая продолжительность жизни по континентам",
        xlab = "Континент", ylab = "Ожидаемая продолжительность жизни (лет)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# Топ-10 стран по приросту ожидаемой продолжительности жизни (1952 против 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "Топ-10: Прирост ожидаемой продолжительности жизни (1952-2007)",
        xlab = "Прирост (лет)",
        col = "#00CC96", border = NA)

## 4. Ожидаемая продолжительность жизни ВОЗ: Исследование на Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Размерность: {who.shape}")
print(f"Колонки: {list(who.columns)}")
print(f"\nПропущенные значения (топ-5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# Развивающиеся и развитые страны: предварительно сгруппированное распределение продолжительности жизни
# Явные координаты столбцов стабильно отрисовываются через браузерный мост Plotly.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Ожидаемая продолжительность жизни: развивающиеся против развитых',
             labels={'Life expectancy': 'Ожидаемая продолжительность жизни (лет)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Образование против ожидаемой продолжительности жизни
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Образование против ожидаемой продолжительности жизни (2014)',
                 labels={'Life expectancy': 'Ожидаемая продолжительность жизни (лет)',
                         'Schooling': 'Продолжительность обучения (лет)'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. Ожидаемая продолжительность жизни ВОЗ: Исследование на R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nСтраны:", length(unique(who$Country)))
cat("\nДиапазон лет:", range(who$Year))

In [ ]:
# Корреляция: смертность взрослого населения против ожидаемой продолжительности жизни
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Смертность взрослых против ожидаемой продолжительности жизни",
     xlab = "Смертность взрослого населения (на 1000)",
     ylab = "Ожидаемая продолжительность жизни (лет)")
legend("topright", legend = c("Развитые", "Развивающиеся"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Простая линейная модель: что предсказывает ожидаемую продолжительность жизни?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Ключевые выводы

- Ожидаемая продолжительность жизни выросла во всем мире, однако между континентами сохраняются значительные различия
- ВВП и уровень образования являются сильными положительными предикторами продолжительности жизни
- Взрослая смертность — наиболее выраженный отрицательный предиктор
- Развивающиеся страны демонстрируют значительно больший разброс показателей